In [5]:
import os
print(os.getcwd())

/home/amahaj56


In [2]:
# FOR JOB

from vllm import LLM, SamplingParams
model_name = "allenai/OLMo-2-1124-7B-Instruct"
llm = LLM(
    model=model_name,
    dtype="float16",
    gpu_memory_utilization=0.85  # safe for single GPU
)
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024
)

import pandas as pd
mapping = pd.read_csv('fasttext_nllb_mapping_98.csv')
mapping = mapping[(mapping.Duplicate_Flag == False) & (mapping.Missing_Flag == False)][['fasttext_code','NLLB_Code']]
mapping_dict = dict(zip(mapping['fasttext_code'], mapping['NLLB_Code']))
mapping_dict.pop('en', None)
mapping_dict.pop('fr', None)
mapping_dict.pop('es', None)
mapping_dict.pop('zh', None)
mapping_dict.pop('hi', None)
mapping_dict.pop('ko', None)
print(len(mapping_dict))

from datasets import *
ds = load_dataset("json", data_files="mgsm_multicolumn_multilang.jsonl", split="train")

ref = load_dataset("juletxara/mgsm", "en", split="test")
references = [ex["answer_number"] for ex in ref]

import re
def extract_final_answer(text: str):
    # Regex matches numbers with optional commas, decimals, and negatives
    numbers = re.findall(r'-?\d[\d,]*\.?\d*', text)
    if not numbers:
        return None
    return numbers[-1]  # return the last number

def compute_accuracy(predictions, gold_answers):
    correct = 0
    total = len(predictions)
    for pred_text, gold in zip(predictions, gold_answers):
        pred_num = extract_final_answer(pred_text)
        if pred_num is not None:
            if pred_num.endswith('.'):
                pred_num = pred_num[:-1]
            pred_num_norm = pred_num.replace(",", "")
        else:
            pred_num_norm = None
        gold_norm = str(gold)
        
        if pred_num_norm == gold_norm:
            correct += 1
    accuracy = correct / total
    return accuracy


# translations
import json
with open("translations.json", "r", encoding="utf-8") as f:
    translations = json.load(f)
multi_lang_prompt = {}

for lang_code, parts in translations.items():
    # Compose the Step-by-Step example by joining steps + answer (mirrors the 'en' example)
    example_steps = (parts.get("steps", "").strip() + " " + parts.get("answer", "").strip()).strip()
    # Build the prompt template with a {} placeholder for the new question
    prompt = f"""{parts.get("instruction", "").strip()}
    Question: {parts.get("question", "").strip()}
    Step-by-Step Answer: {example_steps}
    Final Answer: {parts.get("answer", "").strip()}
    Question: {{}}
    Step-by-Step Answer:  
    Final Answer:"""
    multi_lang_prompt[lang_code] = prompt


results = {}
for lang in list(mapping_dict.keys())[19:]:
    print(lang)
    prompts = [multi_lang_prompt[lang].format(ex[f"question_{lang}"]) for ex in ds]
    batch_size = 32
    all_outputs = []
    
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i + batch_size]
        outputs = llm.generate(batch_prompts, sampling_params)
        for out in outputs:
            all_outputs.append(out.outputs[0].text.strip())
    results[lang] = compute_accuracy(all_outputs, references)
    print(results)

print("="*60)
print(results)

FileNotFoundError: [Errno 2] No such file or directory: 'fasttext_nllb_mapping_98.csv'